In [0]:
%python
# --- Load the Private Key ---
from cryptography.hazmat.primitives import serialization
import base64

# Open your secret key file from the Volume
key_path = "/Volumes/workspace/default/poc_files/rsa_key.p8"
with open(key_path, "rb") as f:
    p_key = serialization.load_pem_private_key(f.read(), password=None)

# Translate it into the format Snowflake's connector needs
pkb = p_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)
pem_private_key = base64.b64encode(pkb).decode("utf-8")

print("✅ Key loaded")

In [0]:
%python
# --- Snowflake --> Bronze ---
from pyspark.sql.functions import current_timestamp, max as smax

# --- Snowflake login details ---
sf_options = {
    "sfUrl":       "yib51904.us-west-2.snowflakecomputing.com",
    "sfUser":      "VGADDAM",                 # 🔧 your username
    "sfDatabase":  "CM360_PERFORMANCE",
    "sfSchema":    "RAW",
    "sfWarehouse": "POC_WH",
    "pem_private_key": pem_private_key
}

bronze_tbl = "poc_incremental_workflow.bronze_incr.cm360_raw"

# --- Find the watermark (newest timestamp already loaded) ---
if spark.catalog.tableExists(bronze_tbl):
    wm = spark.table(bronze_tbl).agg(smax("LOAD_TIMESTAMP")).collect()[0][0]
else:
    wm = None
watermark = "1900-01-01 00:00:00" if wm is None else str(wm)
print(f"🔖 Bronze watermark: {watermark}")

# --- Pull ONLY rows newer than the watermark ---
query = f"""
    SELECT * FROM CM360_PERFORMANCE.RAW.PERFORMANCE_DATA_INCR
    WHERE LOAD_TIMESTAMP > '{watermark}'
"""
new_df = (spark.read.format("snowflake").options(**sf_options)
          .option("query", query).load())

n = new_df.count()
print(f"📥 New rows from Snowflake: {n}")

# --- Append new rows to bronze ---
if n > 0:
    new_df = new_df.withColumn("_ingested_at", current_timestamp())
    (new_df.write.format("delta").mode("append")
        .option("mergeSchema", "true").saveAsTable(bronze_tbl))
    print(f"✅ Appended {n} rows to bronze")
else:
    print("ℹ️ No new data")

In [0]:
%python
# --- Bronze --> Silver ---
from pyspark.sql.functions import col, coalesce, lit, to_timestamp, max as smax
from functools import reduce

bronze_tbl = "poc_incremental_workflow.bronze_incr.cm360_raw"
silver_tbl = "poc_incremental_workflow.silver_incr.cm360_clean"

# --- Silver watermark ---
if spark.catalog.tableExists(silver_tbl):
    swm = spark.table(silver_tbl).agg(smax("LOAD_TIMESTAMP")).collect()[0][0]
else:
    swm = None
silver_wm = "1900-01-01 00:00:00" if swm is None else str(swm)
print(f"🔖 Silver watermark: {silver_wm}")

# --- Take ONLY new bronze rows ---
bronze = spark.table(bronze_tbl).filter(
    col("LOAD_TIMESTAMP") > to_timestamp(lit(silver_wm))
)
new_cnt = bronze.count()
print(f"🧹 New rows to clean: {new_cnt}")

if new_cnt > 0:
    # --- Clean (same as before) ---
    drop_cols = ["BOOKED_IMPRESSIONS","PLANNED_MEDIA_COST",
                 "INTERACTION_RATE","DESIGNATED_MARKET_AREA_DMA"]
    silver_new = bronze.drop(*drop_cols).dropDuplicates()

    metric_cols = ["IMPRESSIONS","CLICKS","CLICK_RATE","TOTAL_CONVERSIONS","MEDIA_COST",
        "VIDEO_FIRST_QUARTILE_COMPLETIONS","VIDEO_MIDPOINTS",
        "VIDEO_THIRD_QUARTILE_COMPLETIONS","VIDEO_COMPLETIONS",
        "VIDEO_INTERACTIONS","CLICK_THROUGH_CONVERSIONS",
        "VIEW_THROUGH_CONVERSIONS","TOTAL_REVENUE"]
    all_zero = reduce(lambda a,b: a & b,
                      [(coalesce(col(c), lit(0)) == 0) for c in metric_cols])
    silver_new = silver_new.filter(~all_zero)

    # --- Append to silver ---
    (silver_new.write.format("delta").mode("append")
        .option("mergeSchema","true").saveAsTable(silver_tbl))
    print(f"✅ Appended {silver_new.count()} cleaned rows to silver")
else:
    print("ℹ️ No new rows to clean")

In [0]:
%sql
--- Silver --> GOld Table 1 ---
CREATE OR REPLACE TABLE poc_incremental_workflow.gold_incr.campaign_performance AS
SELECT CAMPAIGN, CAMPAIGN_ID, ADVERTISER, ADVERTISER_ID,
       PLACEMENT_ID, PLACEMENT, CREATIVE, SITE_CM360, AD,
       SUM(IMPRESSIONS) AS total_impressions,
       SUM(CLICKS) AS total_clicks,
       SUM(TOTAL_CONVERSIONS) AS total_conversions,
       SUM(VIDEO_FIRST_QUARTILE_COMPLETIONS) AS video_q1,
       SUM(VIDEO_MIDPOINTS) AS video_q2_midpoint,
       SUM(VIDEO_THIRD_QUARTILE_COMPLETIONS) AS video_q3
FROM poc_incremental_workflow.silver_incr.cm360_clean
GROUP BY CAMPAIGN, CAMPAIGN_ID, ADVERTISER, ADVERTISER_ID,
         PLACEMENT_ID, PLACEMENT, CREATIVE, SITE_CM360, AD;

In [0]:
%sql
--- Silver --> GOld Table 2 ---
CREATE OR REPLACE TABLE poc_incremental_workflow.gold_incr.campaign_geo_performance AS
SELECT ADVERTISER, ADVERTISER_ID, CAMPAIGN, CAMPAIGN_ID,
       PLACEMENT_ID, PLACEMENT, COUNTRY, STATE_REGION,
       SUM(IMPRESSIONS) AS total_impressions,
       SUM(CLICKS) AS total_clicks,
       SUM(TOTAL_CONVERSIONS) AS total_conversions
FROM poc_incremental_workflow.silver_incr.cm360_clean
GROUP BY ADVERTISER, ADVERTISER_ID, CAMPAIGN, CAMPAIGN_ID,
         PLACEMENT_ID, PLACEMENT, COUNTRY, STATE_REGION;